# 02 — Entity Resolution & Golden Record

Runs the three-tier matching strategy from `ARCHITECTURE.md §8` (deterministic → fuzzy → ML) against the synthetic CRM data, scores it against the known ground truth (`crm_customers_ground_truth.csv`), and builds one example Golden Record with an auditable survivorship decision (`DATA_MODEL.md §4`).

Run `make seed` from the repo root before running this notebook.

In [1]:
from pathlib import Path

import pandas as pd

try:
    import jellyfish
    def name_similarity(a: str, b: str) -> float:
        return jellyfish.jaro_winkler_similarity(a.lower(), b.lower())
except ImportError:
    from difflib import SequenceMatcher
    def name_similarity(a: str, b: str) -> float:
        return SequenceMatcher(None, a.lower(), b.lower()).ratio()

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CRM_PATH = ROOT / "data" / "synthetic" / "crm" / "crm_customers.csv"
GT_PATH = ROOT / "data" / "synthetic" / "crm" / "crm_customers_ground_truth.csv"

if not CRM_PATH.exists() or not GT_PATH.exists():
    raise FileNotFoundError("Run `make seed` from the repo root first — see DATA_MODEL.md.")

crm = pd.read_csv(CRM_PATH)
ground_truth = pd.read_csv(GT_PATH)
known_pairs = set(zip(ground_truth["customer_id"], ground_truth["duplicate_of_customer_id"]))
print(f"{len(crm):,} CRM rows, {len(known_pairs):,} known duplicate pairs")

10,500 CRM rows, 500 known duplicate pairs


## Tier 1 — Deterministic matching

Exact match on `email` or `phone` (skipping nulls/malformed values — those are exactly what tier 2 has to cover instead).

In [2]:
def deterministic_matches(df: pd.DataFrame) -> set[tuple[str, str]]:
    pairs = set()
    for col in ["email", "phone"]:
        valid = df.dropna(subset=[col])
        valid = valid[valid[col].astype(str).str.len() > 4]  # drop malformed phones too
        for _, group in valid.groupby(col):
            ids = group["crm_customer_id"].tolist()
            for i in range(len(ids)):
                for j in range(i + 1, len(ids)):
                    pairs.add((ids[i], ids[j]))
    return pairs

det_pairs = deterministic_matches(crm)
print(f"Deterministic matches found: {len(det_pairs):,}")

Deterministic matches found: 1,023


## Tier 2 — Fuzzy matching

For customers in the same city/state that weren't already matched deterministically, score name similarity (Jaro-Winkler if `jellyfish` is installed, else a `difflib` fallback) and accept pairs above a threshold — this is what catches the name-variant duplicates generators/crm.py injects (`_name_variant`: accent-dropping, abbreviation, case changes).

In [3]:
FUZZY_THRESHOLD = 0.88

fuzzy_pairs = set()
for (city, state), group in crm.groupby(["city", "state"]):
    records = group[["crm_customer_id", "name"]].values.tolist()
    for i in range(len(records)):
        for j in range(i + 1, len(records)):
            id_a, name_a = records[i]
            id_b, name_b = records[j]
            pair = tuple(sorted((id_a, id_b)))
            if pair in det_pairs or (pair[1], pair[0]) in det_pairs:
                continue
            score = name_similarity(str(name_a), str(name_b))
            if score >= FUZZY_THRESHOLD:
                fuzzy_pairs.add(pair)

print(f"Fuzzy matches found (name similarity >= {FUZZY_THRESHOLD}): {len(fuzzy_pairs):,}")

Fuzzy matches found (name similarity >= 0.88): 4,688


## Evaluation against ground truth

The real `mdm/entity_resolution/evaluation/` (Sprint 4-5 acceptance criteria) does exactly this comparison, plus the Tier-3 ML classifier layered on top.

In [4]:
def normalize(pairs):
    return {tuple(sorted(p)) for p in pairs}

predicted = normalize(det_pairs | fuzzy_pairs)
actual = normalize(known_pairs)

true_positives = predicted & actual
false_positives = predicted - actual
false_negatives = actual - predicted

precision = len(true_positives) / len(predicted) if predicted else 0.0
recall = len(true_positives) / len(actual) if actual else 0.0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

print(f"Precision: {precision:.1%}  ({len(true_positives)}/{len(predicted)} predicted pairs correct)")
print(f"Recall:    {recall:.1%}  ({len(true_positives)}/{len(actual)} known duplicates found)")
print(f"F1:        {f1:.1%}")
print(f"\nFalse negatives (missed — candidates for the ML tier, mdm/matching/ml_model.py): {len(false_negatives)}")

Precision: 8.8%  (500/5711 predicted pairs correct)
Recall:    100.0%  (500/500 known duplicates found)
F1:        16.1%

False negatives (missed — candidates for the ML tier, mdm/matching/ml_model.py): 0


## Golden Record example — survivorship in action

Picks one true-positive duplicate pair and shows the field-level survivorship decision from `DATA_MODEL.md §4` (CRM email/phone win; longest non-truncated name wins; most-recent `updated_at` wins on address).

In [5]:
if true_positives:
    example_pair = next(iter(true_positives))
    rows = crm[crm["crm_customer_id"].isin(example_pair)].sort_values("updated_at")
    print("Source records:")
    display(rows[["crm_customer_id", "name", "email", "phone", "city", "state", "updated_at"]])

    survivor = rows.iloc[-1]  # most recently updated wins address, per DATA_MODEL.md §4
    longest_name = rows.loc[rows["name"].str.len().idxmax(), "name"]

    golden_record = {
        "master_customer_id": f"MCID-{example_pair[0]}",
        "canonical_name": longest_name,
        "canonical_email": rows["email"].dropna().iloc[0] if rows["email"].notna().any() else None,
        "canonical_phone": rows["phone"].iloc[0],
        "city": survivor["city"],
        "state": survivor["state"],
        "source_records": rows["crm_customer_id"].tolist(),
    }
    print("\nResulting Golden Record:")
    for k, v in golden_record.items():
        print(f"  {k}: {v}")
else:
    print("No true-positive pair found in this run — try lowering FUZZY_THRESHOLD above.")

Source records:


,crm_customer_id,name,email,phone,city,state,updated_at
964,CRM00000964,Rebeca Silveira,amandacampos@example.net,(084) 5950 8374,São Paulo,SP,2026-05-02T13:05:33
10224,CRM00010224,REBECA SILVEIRA,amandacampos@example.net,(084) 5950 8374,São Paulo,SP,2026-06-16T19:01:26



Resulting Golden Record:
  master_customer_id: MCID-CRM00000964
  canonical_name: Rebeca Silveira
  canonical_email: amandacampos@example.net
  canonical_phone: (084) 5950 8374
  city: São Paulo
  state: SP
  source_records: ['CRM00000964', 'CRM00010224']


## Maps to the real pipeline

- `mdm/matching/deterministic.py`, `mdm/matching/fuzzy.py` — the production versions of tiers 1-2 above.
- `mdm/matching/ml_model.py` — Tier 3 (Random Forest/XGBoost over similarity features), trained specifically on the false negatives this notebook surfaces.
- `mdm/golden_record/survivorship_log/` — every survivorship decision like the one above, logged for every merge, not just this one example.
- `05_customer_graph_analysis.ipynb` — a second, independent way of catching duplicates this notebook's matchers might miss.